# Session 12
- pygame camera
- keyboard arm control

In [ ]:
pip install opencv-python numpy pygame

In [ ]:
import cv2
import numpy as np
from ugot import ugot
got = ugot.UGOT()
got.initialize("192.168.88.1")
ugot_speed = 20
got.open_camera()
import pygame
pygame.init()
screen = None #(width, height)

running = True
x, y = 400, 300
speed = 1
while running:
    frame = got.read_camera_data()
    if not frame:
        continue # skip the rest of the loop if no frame
    nparr = np.frombuffer(frame, np.uint8)
    data = cv2.imdecode(nparr, cv2.IMREAD_COLOR)
    flipped = cv2.flip(data, 1)
    frame_rgb = cv2.cvtColor(flipped, cv2.COLOR_BGR2RGB)
    # create pygame window of matching size
    h, w = frame_rgb.shape[:2]
    if screen is None:
        screen = pygame.display.set_mode((w, h))
        pygame.display.set_caption("UGOT camera")
    surface = pygame.image.frombuffer(frame_rgb.tobytes(),
                (w, h), "RGB")
    screen.blit(surface, (0, 0)) # display camera

    for event in pygame.event.get():
        if event.type == pygame.QUIT:
            running = False
    
    keys = pygame.key.get_pressed()
    if keys[pygame.K_LEFT]:
        x -= speed
    if keys[pygame.K_RIGHT]:
        x += speed
    if keys[pygame.K_UP]:
        y -= speed
    if keys[pygame.K_DOWN]:
        y += speed
    
    # WASD for UGOT movement
    ugot_x, ugot_y, ugot_z = 0, 0, 0
    if keys[pygame.K_a]:
        ugot_x = -ugot_speed
    elif keys[pygame.K_d]:
        ugot_x = ugot_speed
    if keys[pygame.K_s]:
        ugot_y = -ugot_speed
    elif keys[pygame.K_w]:
        ugot_y = ugot_speed
    if keys[pygame.K_q]:
        ugot_z = -ugot_speed
    elif keys[pygame.K_e]:
        ugot_z = ugot_speed
    got.mecanum_move_xyz(ugot_x, ugot_y, ugot_z)

    # screen.fill((255, 255, 255))
    # pygame.draw.circle(screen, (0, 0, 255),
    #                 (x, y), 50)
    pygame.display.flip() # update display

pygame.quit()